# Data Cleaning and Analysis Pipeline

This notebook processes clinical data files, cleans them, detects outliers, and generates summary statistics.

In [1]:
import pandas as pd
from pathlib import Path
import re

## 1. Import Required Libraries

In [2]:
file_names = [
    "HUPA0024P.csv",
    "HUPA0025P.csv",
    "HUPA0026P.csv",
    "HUPA0027P.csv",
    "HUPA0028P.csv"
]

for file in file_names:
    print(file, Path(file).exists())

HUPA0024P.csv True
HUPA0025P.csv True
HUPA0026P.csv True
HUPA0027P.csv True
HUPA0028P.csv True


## 2. Check Input Files Availability

In [3]:
output_folder = Path("cleaned_data")
output_folder.mkdir(exist_ok=True)

## 3. Create Output Directory

In [4]:
def clean_column_name(column_name):
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")
    return column_name

## 4. Define Column Name Cleaning Function

In [5]:
def find_outliers_iqr(df):
    numeric_columns = df.select_dtypes(include="number").columns
    outlier_rows = []

    for column in numeric_columns:
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_count = ((df[column] < lower_bound) | (df[column] > upper_bound)).sum()

        outlier_rows.append({
            "column": column,
            "lower_bound": round(lower_bound, 3),
            "upper_bound": round(upper_bound, 3),
            "outlier_count": int(outlier_count)
        })

    if len(outlier_rows) == 0:
        return pd.DataFrame(columns=[
            "column",
            "lower_bound",
            "upper_bound",
            "outlier_count"
        ])

    outlier_df = pd.DataFrame(outlier_rows)

    return outlier_df.sort_values(
        by="outlier_count",
        ascending=False
    )

## 5. Define Outlier Detection Function (IQR Method)

In [6]:
def clean_dataset(df):
    df = df.copy()

    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")

    df.columns = [clean_column_name(col) for col in df.columns]

    text_columns = df.select_dtypes(include="object").columns

    for column in text_columns:
        df[column] = df[column].astype("string").str.strip()

    for column in df.columns:
        converted = pd.to_numeric(df[column], errors="coerce")

        if converted.notna().sum() > 0:
            df[column] = converted

    numeric_columns = df.select_dtypes(include="number").columns
    df[numeric_columns] = df[numeric_columns].round(3)

    outliers = find_outliers_iqr(df)

    return df, outliers

## 6. Define Data Cleaning Function

In [15]:
cleaned_data = {}
outlier_reports = {}

for file_name in file_names:
    print("=" * 60)
    print("Processing:", file_name)

    df_raw = pd.read_csv(file_name, sep=";")

    print("Original shape:", df_raw.shape)

    df_clean, outliers = clean_dataset(df_raw)

    print("Cleaned shape:", df_clean.shape)

    cleaned_data[file_name] = df_clean
    outlier_reports[file_name] = outliers

    cleaned_file = file_name.replace(".csv", "_cleaned.csv")

    # Save ONLY cleaned file
    df_clean.to_csv(output_folder / cleaned_file, index=False)

    print("Saved:", cleaned_file)

print("DONE")

Processing: HUPA0024P.csv
Original shape: (2902, 8)
Cleaned shape: (2902, 8)
Saved: HUPA0024P_cleaned.csv
Processing: HUPA0025P.csv
Original shape: (4006, 8)
Cleaned shape: (4006, 8)
Saved: HUPA0025P_cleaned.csv
Processing: HUPA0026P.csv
Original shape: (40605, 8)
Cleaned shape: (40605, 8)
Saved: HUPA0026P_cleaned.csv
Processing: HUPA0027P.csv
Original shape: (165306, 8)
Cleaned shape: (165306, 8)
Saved: HUPA0027P_cleaned.csv
Processing: HUPA0028P.csv
Original shape: (25902, 8)
Cleaned shape: (25902, 8)
Saved: HUPA0028P_cleaned.csv
DONE


## 7. Process and Clean All Input Files

In [10]:
for file_name, df in cleaned_data.items():
    print("=" * 60)
    print("Preview:", file_name)
    display(df.head())

Preview: HUPA0024P.csv


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2020-01-20T11:30:00,87.000,8.187,67.364,20.0,0.069,0.0,0.0
1,2020-01-20T11:35:00,86.000,17.988,75.250,107.0,0.069,0.0,0.0
2,2020-01-20T11:40:00,85.000,17.412,76.778,123.0,0.069,0.0,0.0
3,2020-01-20T11:45:00,84.000,10.378,83.667,30.0,0.069,0.0,0.0
4,2020-01-20T11:50:00,85.333,14.529,80.289,74.0,0.069,0.0,0.0


Preview: HUPA0025P.csv


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2020-01-16T13:45:00,108.000,17.583,80.829,54.0,0.0,0.0,0.0
1,2020-01-16T13:50:00,103.667,11.344,74.730,22.0,0.0,0.0,0.0
2,2020-01-16T13:55:00,99.333,13.187,76.444,14.0,0.0,0.0,0.0
3,2020-01-16T14:00:00,95.000,12.762,80.152,17.0,0.0,0.0,0.0
4,2020-01-16T14:05:00,98.333,15.598,74.952,71.0,0.0,0.0,0.0


Preview: HUPA0026P.csv


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2020-05-23T00:15:00,145.000,4.997,85.533,0.0,0.056,0.0,0.0
1,2020-05-23T00:20:00,144.667,5.251,86.226,0.0,0.056,0.0,0.0
2,2020-05-23T00:25:00,144.333,12.026,97.556,112.0,0.056,0.0,0.0
3,2020-05-23T00:30:00,144.000,12.619,98.698,69.0,0.056,0.0,0.0
4,2020-05-23T00:35:00,143.667,10.332,96.323,69.0,0.056,0.0,0.0


Preview: HUPA0027P.csv


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2020-06-26T22:15:00,112.0,22.655,81.000,261.0,0.066,0.0,0.0
1,2020-06-26T22:20:00,105.0,21.162,84.625,200.0,0.066,0.0,0.0
2,2020-06-26T22:25:00,98.0,25.643,89.222,292.0,0.066,0.0,0.0
3,2020-06-26T22:30:00,91.0,12.572,80.028,68.0,0.066,0.0,0.0
4,2020-06-26T22:35:00,91.0,6.722,75.108,0.0,0.066,0.0,0.0


Preview: HUPA0028P.csv


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2022-02-17T13:50:00,78.333,13.581,86.040,198.0,0.0,0.0,0.0
1,2022-02-17T13:55:00,77.667,28.421,121.809,374.0,0.0,0.0,0.0
2,2022-02-17T14:00:00,77.000,15.919,101.689,96.0,0.0,0.0,0.0
3,2022-02-17T14:05:00,76.667,8.814,91.309,0.0,0.0,0.0,0.0
4,2022-02-17T14:10:00,76.333,8.454,93.474,0.0,0.0,0.0,0.0


## 8. Display Preview of Cleaned Data

In [11]:
for file_name, report in outlier_reports.items():
    print("=" * 60)
    print("Outliers:", file_name)
    display(report)

Outliers: HUPA0024P.csv


,column,lower_bound,upper_bound,outlier_count
3,steps,-138.000,230.000,150
5,bolus_volume_delivered,0.000,0.000,21
6,carb_input,0.000,0.000,14
1,calories,-15.334,40.933,5
0,glucose,-46.375,375.959,0
2,heart_rate,20.299,126.769,0
4,basal_rate,-0.104,0.173,0


Outliers: HUPA0025P.csv


,column,lower_bound,upper_bound,outlier_count
3,steps,-31.500,52.500,674
1,calories,-2.055,22.332,399
5,bolus_volume_delivered,0.000,0.000,130
2,heart_rate,44.241,111.942,52
0,glucose,3.168,220.500,23
6,carb_input,0.000,0.000,20
4,basal_rate,-0.074,0.258,0


Outliers: HUPA0026P.csv


,column,lower_bound,upper_bound,outlier_count
3,steps,-81.000,135.000,5426
1,calories,-1.018,12.942,4820
5,bolus_volume_delivered,0.000,0.000,334
0,glucose,-40.500,360.832,212
2,heart_rate,43.951,115.282,95
6,carb_input,0.000,0.000,46
4,basal_rate,-0.084,0.140,0


Outliers: HUPA0027P.csv


,column,lower_bound,upper_bound,outlier_count
3,steps,0.000,0.000,38057
1,calories,1.169,14.465,22933
2,heart_rate,31.871,116.388,5528
0,glucose,3.356,250.786,2680
5,bolus_volume_delivered,0.000,0.000,1876
6,carb_input,0.000,0.000,1531
4,basal_rate,-0.099,0.165,0


Outliers: HUPA0028P.csv


,column,lower_bound,upper_bound,outlier_count
3,steps,0.000,0.000,5526
1,calories,2.204,8.320,3910
2,heart_rate,41.072,106.157,852
0,glucose,31.744,222.553,255
5,bolus_volume_delivered,0.000,0.000,223
6,carb_input,0.000,0.000,217
4,basal_rate,-0.063,0.105,0


## 9. Display Outlier Reports

In [12]:
summary = []

for file_name, df in cleaned_data.items():
    outliers = outlier_reports[file_name]

    summary.append({
        "file_name": file_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "numeric_columns": len(df.select_dtypes(include="number").columns),
        "total_outliers": outliers["outlier_count"].sum()
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

,file_name,rows,columns,numeric_columns,total_outliers
0,HUPA0024P.csv,2902,8,7,190
1,HUPA0025P.csv,4006,8,7,1298
2,HUPA0026P.csv,40605,8,7,10933
3,HUPA0027P.csv,165306,8,7,72605
4,HUPA0028P.csv,25902,8,7,10983


## 10. Generate Data Quality Summary

In [13]:
clinical_summary = []

for file_name, df in cleaned_data.items():

    summary = {
        "file_name": file_name
    }

    # Glucose features
    if "glucose" in df.columns:
        summary["glucose_mean"] = round(df["glucose"].mean(), 3)
        summary["glucose_std"] = round(df["glucose"].std(), 3)
        summary["glucose_min"] = round(df["glucose"].min(), 3)
        summary["glucose_max"] = round(df["glucose"].max(), 3)

    # Heart rate features
    if "heart_rate" in df.columns:
        summary["hr_mean"] = round(df["heart_rate"].mean(), 3)
        summary["hr_std"] = round(df["heart_rate"].std(), 3)
        summary["hr_max"] = round(df["heart_rate"].max(), 3)

    # Steps
    if "steps" in df.columns:
        summary["steps_sum"] = round(df["steps"].sum(), 3)

    # Carbs
    if "carb_input" in df.columns:
        summary["carbs_sum"] = round(df["carb_input"].sum(), 3)

    # Bolus
    if "bolus_volume_delivered" in df.columns:
        summary["bolus_sum"] = round(df["bolus_volume_delivered"].sum(), 3)

    # Basal
    if "basal_rate" in df.columns:
        summary["basal_mean"] = round(df["basal_rate"].mean(), 3)

    clinical_summary.append(summary)

clinical_features_df = pd.DataFrame(clinical_summary)

display(clinical_features_df)

,file_name,glucose_mean,glucose_std,glucose_min,glucose_max,hr_mean,hr_std,hr_max,steps_sum,carbs_sum,bolus_sum,basal_mean
0,HUPA0024P.csv,166.944,66.546,42.0,359.0,73.794,15.898,117.597,163403.0,76.50,71.00,0.042
1,HUPA0025P.csv,113.846,40.053,40.0,245.0,79.110,13.710,195.615,111473.0,58.50,360.65,0.094
2,HUPA0026P.csv,162.985,68.665,40.0,422.0,80.285,12.016,156.909,1929063.0,176.20,2993.00,0.036
3,HUPA0027P.csv,130.830,46.267,40.0,397.0,76.785,16.567,178.353,4380913.0,5161.25,12841.10,0.040
4,HUPA0028P.csv,128.436,35.451,40.0,296.0,75.643,13.320,149.273,492727.0,711.50,778.00,0.017


## 11. Calculate Clinical Features Summary

In [14]:
clinical_features_df.to_csv(
    output_folder / "clinical_features_summary.csv",
    index=False
)

print("Clinical feature summary saved.")

Clinical feature summary saved.


## 12. Save Clinical Features Summary to CSV